# ERP–Blockchain Adaptive Execution — ADDENDUM to v5 (analysis only, design unchanged)

**This notebook does NOT change the controller, the thresholds, the protocol or the design hash.** It re-uses the exact v5 code and adds these analyses (reproducibility, analytical validation, sensitivity incl. completed TPS, overload throughput, matched off-chain frontier, bursty ERP workload, capacity dip with recovery, priority fairness, decision divergence) that answer reviewer comments:

| # | Addition | Answers |
|---|---|---|
| 1 | **Reproducibility check**: re-runs a slice of the v5 simulation and compares with your stored `sim_raw_results.csv` | R1 #1 (consistency between reported results and implementation) |
| 2 | **Analytical validation of the simulator**: at vanishing load the latency must equal the *residual-life* distribution of the real block intervals (exact theory); KS test + overload-saturation test | R2 #1 ("independently validated execution model") |
| 3 | **Sensitivity at ρ = 0.5, 0.7 (and 0.85)** over 11 assumption variants (off-loadable share, anchor size, class mix, batch max-wait, aging) | R1 #2 (conclusions under alternative assumptions) |
| 4 | **Fig 5 / Fig 6 for the models actually selected** (permutation importance + residual diagnostics), incl. a negative control for the unpredictable interval task | R1 #5 (relation between selected model and the figures) |
| 5 | *Post-hoc / exploratory* **value-of-information table** (ORACLE headroom) — clearly labelled, not a pre-specified claim | transparency |

### Before running
1. Run cell 0 and cell 1 (this creates `/content/results_v5_2_1_addendum`).
2. **Upload into `/content/results_v5_2_1_addendum/`** these files from your `results_v5_2_1.zip`:
   `blocks_2020-01-01_2021-04-01.parquet`, `table_tuning_validation.csv`, `sim_raw_results.csv`, `RESULTS_SUMMARY.json` (and optionally `api_checkpoint.pkl`).
3. Run the remaining cells top to bottom (≈ 60–75 min).

In [ ]:
# 0) INSTALL (Colab)
!pip -q install pyarrow requests

In [ ]:
# 1) IMPORTS + CONFIG
import os, sys, json, time, warnings, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sps
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
DATA_SOURCE  = "api"        # "api" = FREE public Bitcoin block API (no account, no key)  |  "bigquery" = needs a Google Cloud project
PROJECT_ID   = ""           # only used if DATA_SOURCE == "bigquery"  (your Google Cloud Project ID)
YEAR_START, YEAR_END = "2020-01-01", "2021-04-01"
MIN_BLOCKS   = 64000                      # 2020-01-01 -> 2021-04-01 has ~66,000 blocks
SEED         = 42
TRAIN_FRAC, VAL_FRAC = 0.60, 0.20         # used ONLY in QUICK mode
T_TRAIN_END, T_VAL_END = "2020-10-01", "2021-01-01"   # train | validation (tuning) | CONFIRMATION TEST = 2021-01-01 .. 2021-03-31
EMBARGO      = 6                          # purge: demand label looks 6 blocks ahead
QUICK        = False
OUT_DIR      = "/content/results_v5_2_1_addendum"
os.makedirs(OUT_DIR, exist_ok=True)

# closed loop (walk-forward) on the DEMAND forecaster
CHUNK, WINDOW, PERIODIC_EVERY = 144, 20000, 20
PH_DELTA, PH_LAMBDA = 0.05, 1.0

# simulator (every number below is an explicit, testable assumption)
SIM = dict(
    CAP=100.0, CLASS_P=[0.10, 0.60, 0.30], CLASS_W=np.array([3.0, 1.0, 0.0]),   # HIGH, NORMAL, LOW
    T_AGE=1800.0, ANCHOR=1.0, OFFLOADABLE_P=0.5, MAX_WAIT_PARTIAL=1200.0,
    SIZE_SIGMA=0.8, SLA=3600.0, WARMUP_FRAC=0.05, DT_MEAN=600.0,
)
RHOS, N_SEEDS = [0.5, 0.7, 0.85, 0.95], 5
TUNE_RHOS, TUNE_SEEDS = [0.3, 0.6, 0.9], 1 # tuned on SEVERAL loads so the controller generalises (0.2 and 0.95 are unseen)
SLA_FLOOR = 0.90                           # pre-specified constraint: every class must keep SLA hit-rate >= 90 %
GRID_LO, GRID_HI = [0.0, 0.4, 0.8], [0.6, 1.0, 1.5, 2.5]
SENS_SEEDS = 2
EPS_TOL = 0.05          # eps-constraint: among configs within 5 % of the best latency objective pick the one using the LEAST off-chain
H_FWD = 6               # forecast horizon (blocks) = length of the demand window the forecaster predicts
RECAL_WINDOW, COOLDOWN = 6000, 2   # recalibration trains on the most recent blocks; min cycles between recalibrations
DRIFT_SCALE = 1.35      # drift-injection stress test: workload x1.35 from the middle of the test period
if QUICK:
    RHOS, N_SEEDS, TUNE_SEEDS, SENS_SEEDS, TUNE_RHOS = [0.9], 2, 1, 1, [0.3, 0.9]
    GRID_LO, GRID_HI = [0.0], [0.6, 1.5]
SIM["H"] = H_FWD
CLS = ["HIGH", "NORMAL", "LOW"]
import hashlib
CODE_VERSION = "v5.2.1"   # bump on ANY change that can alter results
NORMALISATION = "demand scale = TRAINING-period mean (fixed reference for validation and confirmation)"
CONFIG_HASH = hashlib.sha256(json.dumps(dict(CODE_VERSION=CODE_VERSION, NORMALISATION=NORMALISATION,
    SIM={k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in SIM.items()}, RHOS=RHOS, N_SEEDS=N_SEEDS, TUNE_RHOS=TUNE_RHOS,
    TUNE_SEEDS=TUNE_SEEDS, GRID_LO=GRID_LO, GRID_HI=GRID_HI, EPS_TOL=EPS_TOL, SLA_FLOOR=SLA_FLOOR, H_FWD=H_FWD, CHUNK=CHUNK,
    WINDOW=WINDOW, RECAL_WINDOW=RECAL_WINDOW, COOLDOWN=COOLDOWN, PH=(PH_DELTA, PH_LAMBDA), EMBARGO=EMBARGO, SEED=SEED,
    SPLIT=(YEAR_START, T_TRAIN_END, T_VAL_END, YEAR_END)), sort_keys=True).encode()).hexdigest()[:16]
print("Config OK | QUICK =", QUICK)
print("CODE VERSION:", CODE_VERSION, "|", NORMALISATION)
print("DESIGN FREEZE hash:", CONFIG_HASH, " <- record it; it must be identical in every run that you report")

In [ ]:
# A0) INPUT FILES  (asks you to upload them if they are missing; you can upload results_v5_2_1.zip directly)
import zipfile, io
REQUIRED = ["table_tuning_validation.csv", "sim_raw_results.csv", "RESULTS_SUMMARY.json"]
def handle_uploads(up, dest):
    for name, data in up.items():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as z: z.extractall(dest)
            print("extracted", name, "->", dest)
        else:
            base = os.path.basename(name).split(" (")[0] + (os.path.splitext(name)[1] if " (" in name and not os.path.splitext(name.split(" (")[0])[1] else "")
            open(f"{dest}/{base}", "wb").write(data); print("saved", base)
missing = [f for f in REQUIRED if not os.path.exists(f"{OUT_DIR}/{f}")]
if missing:
    print("Missing input files:", missing)
    print("In the dialog below select your results_v5_2_1.zip (easiest) OR the individual files from it (also the .parquet if you have it).")
    try:
        from google.colab import files
        handle_uploads(files.upload(), OUT_DIR)
    except ImportError:
        print("Not running in Colab: copy the files into", OUT_DIR, "manually.")
    missing = [f for f in REQUIRED if not os.path.exists(f"{OUT_DIR}/{f}")]
assert not missing, f"Still missing {missing} in {OUT_DIR}. Upload them (or the whole results_v5_2_1.zip) and re-run this cell."
print("Input files OK:", sorted(os.listdir(OUT_DIR)))

In [ ]:
# 2) DATA EXTRACTION + COVERAGE GUARD  (this is what prevents the "15,000 blocks" mistake)
CACHE = f"{OUT_DIR}/blocks_{YEAR_START}_{YEAR_END}.parquet"

# ---------- FREE source: public Esplora-compatible block APIs (no account / no key / no billing) ----------
API_BASES = ["https://mempool.space/api", "https://blockstream.info/api"]
H_LO, H_HI = 610_000, 678_000         # generous height range that contains all of 2020; filtered by timestamp afterwards
if QUICK: H_HI = H_LO + 13_000

def fetch_from_api(workers=4):
    import requests, pickle, concurrent.futures as cf
    starts = list(range(H_HI, H_LO, -10))          # every request returns >= 10 blocks going DOWN from `start` -> full coverage
    ck = f"{OUT_DIR}/api_checkpoint.pkl"
    done = pickle.load(open(ck, "rb")) if os.path.exists(ck) else {}
    todo = [h for h in starts if h not in done]
    print(f"API fetch: {len(starts)} pages total, {len(done)} already cached, {len(todo)} to fetch (resumable)")
    sess = requests.Session(); sess.headers.update({"User-Agent": "research-notebook/1.0"})
    def one(h):
        for attempt in range(8):
            base = API_BASES[(attempt // 2) % len(API_BASES)]     # 2 tries per host, then switch host
            try:
                r = sess.get(f"{base}/blocks/{h}", timeout=30)
                if r.status_code == 200:
                    return h, [(b["height"], b["timestamp"], b["tx_count"], b["size"], b["weight"]) for b in r.json()]
            except Exception:
                pass
            time.sleep(min(30, 2 ** attempt))                     # back off on 429 / network errors
        return h, None
    failed, t0 = 0, time.time()
    with cf.ThreadPoolExecutor(max_workers=workers) as ex:
        for i, (h, rows) in enumerate(ex.map(one, todo), 1):
            if rows is None: failed += 1
            else: done[h] = rows
            if i % 250 == 0:
                pickle.dump(done, open(ck, "wb")); print(f"  {i}/{len(todo)} pages  ({time.time()-t0:.0f}s)")
    pickle.dump(done, open(ck, "wb"))
    if failed:
        raise RuntimeError(f"{failed} pages failed after retries. Just re-run this cell: it RESUMES from the checkpoint. "
                           "If it keeps failing, wait a few minutes (rate limit) or set DATA_SOURCE='bigquery'.")
    rows = [r for v in done.values() for r in v]
    r_ = pd.DataFrame(rows, columns=["number", "ts", "transaction_count", "size", "weight"]).drop_duplicates("number")
    r_["block_timestamp"] = pd.to_datetime(r_["ts"], unit="s", utc=True)
    r_ = r_[(r_.block_timestamp >= pd.Timestamp(YEAR_START, tz="UTC")) & (r_.block_timestamp < pd.Timestamp(YEAR_END, tz="UTC"))]
    r_ = r_.sort_values("number").reset_index(drop=True)[["number", "block_timestamp", "transaction_count", "size", "weight"]]
    r_.to_parquet(CACHE); return r_

# ---------- OPTIONAL source: BigQuery (free tier / sandbox, but needs a Google Cloud project you have access to) ----------
def resolve_project_id():
    pid = (PROJECT_ID or "").strip()
    bad = (not pid) or ("ENTER" in pid.upper()) or ("YOUR" in pid.upper()) or pid.startswith("<")
    if bad:
        try:
            import google.auth
            _, pid = google.auth.default()
        except Exception:
            pid = None
    if not pid:
        pid = input("Paste your Google Cloud PROJECT ID (not the project name): ").strip()
    print("Using BigQuery project:", pid); return pid

def fetch_from_bigquery():
    from google.colab import auth
    auth.authenticate_user()
    import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'google-cloud-bigquery', 'db-dtypes'])
    from google.cloud import bigquery
    client = bigquery.Client(project=resolve_project_id())
    q = f"""
    SELECT number, timestamp AS block_timestamp, transaction_count, size, weight
    FROM `bigquery-public-data.crypto_bitcoin.blocks`
    WHERE timestamp >= TIMESTAMP('{YEAR_START}') AND timestamp < TIMESTAMP('{YEAR_END}')
    ORDER BY number
    """
    try:
        r = client.query(q).to_dataframe()
    except Exception as e:
        raise RuntimeError("BigQuery failed (project id / permission / API not enabled). Easiest fix: set DATA_SOURCE='api'. Original: " + str(e))
    r.to_parquet(CACHE); return r

fetch = fetch_from_api if DATA_SOURCE == "api" else fetch_from_bigquery

def coverage_problems(r):
    msgs = []; ts = pd.to_datetime(r["block_timestamp"], utc=True)
    if len(r) < MIN_BLOCKS: msgs.append(f"only {len(r)} blocks (< {MIN_BLOCKS})")
    if ts.min() > pd.Timestamp(YEAR_START, tz="UTC") + pd.Timedelta(days=1): msgs.append(f"starts late: {ts.min()}")
    if ts.max() < pd.Timestamp(YEAR_END, tz="UTC") - pd.Timedelta(days=3): msgs.append(f"ends early: {ts.max()}")
    nn = np.sort(r["number"].values)
    if int(r["number"].duplicated().sum()): msgs.append("duplicate block numbers")
    if int((np.diff(nn) != 1).sum()): msgs.append("gaps in block numbers")
    return msgs

used_cache = os.path.exists(CACHE)
raw = pd.read_parquet(CACHE) if used_cache else fetch()
if not QUICK:
    probs = coverage_problems(raw)
    if probs and used_cache:
        print("STALE/PARTIAL CACHE ->", probs, "\nRe-fetching ...")
        os.remove(CACHE); raw = fetch(); probs = coverage_problems(raw)
    assert not probs, f"Data coverage check FAILED: {probs}"
    print("Coverage guard PASSED:", len(raw), "blocks,", raw.block_timestamp.min(), "->", raw.block_timestamp.max())
else:
    raw = raw.iloc[:12000].copy(); print("QUICK mode: coverage guard skipped")
display(raw.head())

In [ ]:
# 3) FEATURES (leak-free) + PURGED CHRONOLOGICAL SPLIT + DATASET AUDIT
# Unit = one Bitcoin block.  Row i uses ONLY information available after block i-1.
TARGET_INT = "block_time"   # inter-block timestamp interval (miner-reported)  -> Task A
TARGET_DEM = "demand"       # mean tx_count of blocks i..i+5 (workload intensity, forward window) -> Task B

def build_base_features(raw):
    d = raw.sort_values("number").reset_index(drop=True).copy()
    d["block_timestamp"] = pd.to_datetime(d["block_timestamp"], utc=True)
    d["dt_raw"] = d["block_timestamp"].diff().dt.total_seconds()
    n_neg = int((d["dt_raw"] < 0).sum())
    d[TARGET_INT] = d["dt_raw"].clip(lower=0)
    tx, sz, wt, bt = d["transaction_count"], d["size"], d["weight"], d[TARGET_INT]
    d[TARGET_DEM] = tx.iloc[::-1].rolling(6).mean().iloc[::-1]          # forward window i..i+5
    d["prev_tx"], d["prev_size"], d["prev_weight"] = tx.shift(1), sz.shift(1), wt.shift(1)
    d["hour"]      = d["block_timestamp"].shift(1).dt.hour
    d["dayofweek"] = d["block_timestamp"].shift(1).dt.dayofweek
    d["tx_ma_5"]     = tx.shift(1).rolling(5).mean()
    d["tx_std_5"]    = tx.shift(1).rolling(5).std()
    d["size_ma_5"]   = sz.shift(1).rolling(5).mean()
    d["weight_ma_5"] = wt.shift(1).rolling(5).mean()
    d["tx_ma_36"]    = tx.shift(1).rolling(36).mean()
    d["tx_ma_144"]   = tx.shift(1).rolling(144, min_periods=36).mean()
    d["prev_block_time"]       = bt.shift(1)
    d["prev_block_time_ma_3"]  = bt.shift(1).rolling(3).mean()
    d["prev_block_time_std_3"] = bt.shift(1).rolling(3).std()
    d["tx_pct_change"]     = tx.pct_change().replace([np.inf, -np.inf], np.nan).shift(1)
    d["weight_pct_change"] = wt.pct_change().replace([np.inf, -np.inf], np.nan).shift(1)
    return d, n_neg

base, n_neg_dt = build_base_features(raw)
BASE_COLS = ["prev_tx","prev_size","prev_weight","hour","dayofweek","tx_ma_5","tx_std_5","size_ma_5","weight_ma_5",
             "tx_ma_36","tx_ma_144","prev_block_time","prev_block_time_ma_3","prev_block_time_std_3",
             "tx_pct_change","weight_pct_change"]
df = base.dropna(subset=BASE_COLS + [TARGET_INT, TARGET_DEM]).reset_index(drop=True)
ts_ = df["block_timestamp"]
if QUICK:
    n = len(df); i_tr = int(n * TRAIN_FRAC); i_va = int(n * (TRAIN_FRAC + VAL_FRAC))
else:                                              # DATE-BASED, out-of-time split
    i_tr = int((ts_ < pd.Timestamp(T_TRAIN_END, tz="UTC")).sum()); i_va = int((ts_ < pd.Timestamp(T_VAL_END, tz="UTC")).sum()); n = len(df)
    assert i_tr > 30000 and i_va - i_tr > 10000 and n - i_va > 10000, f"unexpected split sizes {i_tr}, {i_va - i_tr}, {n - i_va}"
a_tr, a_va = i_tr - EMBARGO, i_va - EMBARGO           # purged boundaries (train: [:a_tr], val: [i_tr:a_va], test: [i_va:])
tr = df.iloc[:a_tr]
NORM = dict(tx_mean=float(tr.prev_tx.mean()), tx_std=float(tr.prev_tx.std()),
            tx_max=float(tr.prev_tx.max()), w_max=float(tr.prev_weight.max()))
df["burst_flag"]     = (df["prev_tx"] > NORM["tx_mean"] + NORM["tx_std"]).astype(int)
df["queue_pressure"] = 0.5 * df["prev_tx"] / NORM["tx_max"] + 0.5 * df["prev_weight"] / NORM["w_max"]

FEATURES_INT = ["prev_tx","prev_size","prev_weight","hour","dayofweek","tx_ma_5","tx_std_5","size_ma_5","weight_ma_5",
                "prev_block_time","prev_block_time_ma_3","prev_block_time_std_3","burst_flag","queue_pressure",
                "tx_pct_change","weight_pct_change"]
FEATURES_DEM = FEATURES_INT + ["tx_ma_36", "tx_ma_144"]
SIM["DT_MEAN"] = float(df[TARGET_INT].iloc[:a_tr].mean())
DEMAND_SCALE = float(df[TARGET_DEM].iloc[:a_tr].mean())   # TRAINING-only reference for normalising demand/forecasts (no val/test statistic enters the controller)

audit = dict(
    protocol="train 2020-01..09 | validation 2020-10..12 (all tuning) | confirmation test 2021-01..03 (untouched)", config_hash=CONFIG_HASH,
    observation_unit="one Bitcoin block", code_version=CODE_VERSION, demand_scale_train_mean=DEMAND_SCALE, data_source=DATA_SOURCE, data_source_detail=("public Esplora-compatible block APIs (mempool.space primary, blockstream.info fallback)" if DATA_SOURCE == "api" else "Google BigQuery bigquery-public-data.crypto_bitcoin.blocks"), block_height_min=int(raw["number"].min()), block_height_max=int(raw["number"].max()), duplicate_heights=int(raw["number"].duplicated().sum()), date_range=[YEAR_START, YEAR_END],
    blocks_fetched=int(len(raw)), rows_after_feature_construction=int(n), rows_dropped=int(len(raw) - n),
    n_features_interval_task=len(FEATURES_INT), n_features_demand_task=len(FEATURES_DEM),
    embargo_blocks=EMBARGO, n_train=int(a_tr), n_val=int(a_va - i_tr), n_test=int(n - i_va),
    train_range=[str(df.block_timestamp.iloc[0]), str(df.block_timestamp.iloc[a_tr-1])],
    val_range=[str(df.block_timestamp.iloc[i_tr]), str(df.block_timestamp.iloc[a_va-1])],
    test_range=[str(df.block_timestamp.iloc[i_va]), str(df.block_timestamp.iloc[-1])],
    negative_intervals_clipped_to_0=n_neg_dt,
    interval_mean_s=float(df[TARGET_INT].mean()), interval_cv=float(df[TARGET_INT].std() / df[TARGET_INT].mean()),
    interval_lag1_autocorr_train=float(df[TARGET_INT].iloc[:a_tr].autocorr(1)),
    demand_lag1_autocorr_train=float(df[TARGET_DEM].iloc[:a_tr].autocorr(1)),
    demand_mean_train_val_test=[float(df[TARGET_DEM].iloc[:a_tr].mean()), float(df[TARGET_DEM].iloc[i_tr:a_va].mean()),
                                float(df[TARGET_DEM].iloc[i_va:].mean())],
    norm_constants_from_train=NORM,
)
json.dump(audit, open(f"{OUT_DIR}/dataset_audit.json", "w"), indent=2)
print(json.dumps(audit, indent=2))

In [ ]:
# 4) TWO PREDICTION TASKS (selection on VALIDATION, test touched once)
def make_models():
    ne = 100 if QUICK else 300
    wrap = lambda m: Pipeline([("sc", StandardScaler()), ("m", m)])
    return {
        "RandomForest": wrap(RandomForestRegressor(n_estimators=ne, max_depth=12, min_samples_split=5, random_state=SEED, n_jobs=-1)),
        "GradientBoosting": wrap(GradientBoostingRegressor(n_estimators=ne, learning_rate=0.03, max_depth=3, random_state=SEED)),
        "MLP": wrap(MLPRegressor(hidden_layer_sizes=(128, 64), activation="relu", solver="adam", learning_rate_init=0.0008,
                                 max_iter=700, early_stopping=True, random_state=SEED)),
        "Ridge": wrap(Ridge(alpha=1.0)),
    }
SL = dict(tr=slice(0, a_tr), va=slice(i_tr, a_va), te=slice(i_va, n))

def metrics(yt, yp):
    return dict(MAE=mean_absolute_error(yt, yp), RMSE=float(np.sqrt(mean_squared_error(yt, yp))), R2=r2_score(yt, yp))

def run_task(tag, target, feats, baselines, model_names):
    yy, Xf = df[target], df[feats]
    PV, PT, fitted, rows = {}, {}, {}, []
    for bn, fn in baselines.items():
        PV[bn], PT[bn] = fn(Xf.iloc[SL["va"]]), fn(Xf.iloc[SL["te"]])
    for nm in model_names:
        m = make_models()[nm]; t0 = time.perf_counter()
        m.fit(Xf.iloc[SL["tr"]], yy.iloc[SL["tr"]]); fitted[nm] = m
        PV[nm], PT[nm] = m.predict(Xf.iloc[SL["va"]]), m.predict(Xf.iloc[SL["te"]])
        print(f"[{tag}] {nm}: trained in {time.perf_counter() - t0:.1f}s")
    for nm in PV:
        mv, mt = metrics(yy.iloc[SL["va"]], PV[nm]), metrics(yy.iloc[SL["te"]], PT[nm])
        rows.append(dict(Model=nm, **{f"val_{k}": v for k, v in mv.items()}, **{f"test_{k}": v for k, v in mt.items()}))
    table = pd.DataFrame(rows).set_index("Model")
    sel = table.loc[model_names].sort_values("val_RMSE").index[0]
    best_base = table.loc[list(baselines)].sort_values("val_RMSE").index[0]
    return dict(tag=tag, target=target, feats=feats, table=table, SEL=sel, best_base=best_base, fitted=fitted,
                PV=PV, PT=PT, ref_mae=float(table.loc[sel, "val_MAE"]),
                y_va=yy.iloc[SL["va"]].values, y_te=yy.iloc[SL["te"]].values)

seas = df.iloc[:a_tr].groupby(df["hour"].iloc[:a_tr].astype(int))[TARGET_DEM].mean()
BASE_INT = {"Baseline: train mean": lambda X: np.full(len(X), df[TARGET_INT].iloc[:a_tr].mean()),
            "Baseline: previous interval": lambda X: X["prev_block_time"].values}
BASE_DEM = {"Baseline: train mean": lambda X: np.full(len(X), df[TARGET_DEM].iloc[:a_tr].mean()),
            "Baseline: persistence (5-block mean)": lambda X: X["tx_ma_5"].values,
            "Baseline: 36-block mean": lambda X: X["tx_ma_36"].values,
            "Baseline: hour-of-day mean": lambda X: X["hour"].astype(int).map(seas).fillna(seas.mean()).values}

TA = run_task("A: interval", TARGET_INT, FEATURES_INT, BASE_INT, ["RandomForest", "GradientBoosting", "MLP"])
TB = run_task("B: demand",   TARGET_DEM, FEATURES_DEM, BASE_DEM, ["Ridge", "RandomForest", "GradientBoosting", "MLP"])
for T in (TA, TB):
    print(f"\n===== Task {T['tag']}  (target = {T['target']}) =====")
    display(T["table"].round(4)); T["table"].to_csv(f"{OUT_DIR}/table_prediction_{T['tag'][0]}.csv")
    print("Selected on validation:", T["SEL"], "| best baseline on validation:", T["best_base"])

In [ ]:
# 5) PREDICTION STATISTICS THAT RESPECT TEMPORAL DEPENDENCE
def block_bootstrap_ci(a, b, block=144, n_boot=3000, seed=SEED):
    rng = np.random.default_rng(seed); a, b = np.asarray(a, float), np.asarray(b, float)
    N = len(a); k = int(np.ceil(N / block)); out = np.empty(n_boot)
    for i in range(n_boot):
        starts = rng.integers(0, N - block + 1, k)
        idx = (starts[:, None] + np.arange(block)[None, :]).ravel()[:N]
        out[i] = 100 * (a[idx].mean() - b[idx].mean()) / a[idx].mean()
    return np.percentile(out, [2.5, 97.5])

def ljung_box(res, lags=(10, 50)):
    res = np.asarray(res, float) - np.mean(res); N = len(res)
    acf = np.array([np.sum(res[k:] * res[:N - k]) / np.sum(res**2) for k in range(1, max(lags) + 1)])
    return {L: dict(Q=float(N*(N+2)*np.sum(acf[:L]**2/(N-np.arange(1, L+1)))),
                    p=float(1 - sps.chi2.cdf(N*(N+2)*np.sum(acf[:L]**2/(N-np.arange(1, L+1))), L))) for L in lags}

PRED_STATS = {}
for T in (TA, TB):
    e_sel = np.abs(T["y_te"] - T["PT"][T["SEL"]]); rows = []
    for bn in [b for b in T["PT"] if b.startswith("Baseline")]:
        e_b = np.abs(T["y_te"] - T["PT"][bn]); lo, hi = block_bootstrap_ci(e_b, e_sel)
        rows.append(dict(task=T["tag"], model=T["SEL"], vs=bn, MAE_improvement_pct=100*(e_b.mean()-e_sel.mean())/e_b.mean(),
                         CI95_low=lo, CI95_high=hi, significant=bool(lo > 0)))
    PRED_STATS[T["tag"]] = pd.DataFrame(rows)
    print(f"\n--- Task {T['tag']}: {T['SEL']} vs baselines (moving-block bootstrap, block=144) ---")
    display(PRED_STATS[T["tag"]].round(3))
    print("Ljung-Box on test residuals:", json.dumps(ljung_box(T["y_te"] - T["PT"][T["SEL"]])))
pd.concat(PRED_STATS.values()).to_csv(f"{OUT_DIR}/table_prediction_block_bootstrap.csv", index=False)

In [ ]:
# 6) CLOSED LOOP on the demand forecaster: walk-forward + Page-Hinkley + recalibration (real 2020 test period)
class PageHinkley:
    def __init__(self, delta=0.05, lam=1.0): self.delta, self.lam = delta, lam; self.reset()
    def reset(self): self.n, self.mean, self.cum, self.min = 0, 0.0, 0.0, 0.0
    def update(self, x):
        self.n += 1; self.mean += (x - self.mean) / self.n
        self.cum += x - self.mean - self.delta; self.min = min(self.min, self.cum)
        return (self.cum - self.min) > self.lam

def fit_initial(T, dfx=None):
    dfx = df if dfx is None else dfx; lo0 = max(0, a_va - WINDOW); pipe = make_models()[T["SEL"]]
    pipe.fit(dfx[T["feats"]].iloc[lo0:a_va], dfx[T["target"]].iloc[lo0:a_va]); return pipe     # train+val (purged)

def walk_forward(T, mode, init_pipe, dfx=None):
    """mode: 'frozen' (no feedback) | 'periodic' | 'drift' (feedback-triggered recalibration on the latest RECAL_WINDOW blocks)."""
    dfx = df if dfx is None else dfx; Xf, yy = dfx[T["feats"]], dfx[T["target"]]
    pipe = copy.deepcopy(init_pipe); ph = PageHinkley(PH_DELTA, PH_LAMBDA); n_te = n - i_va
    preds, chunk_mae, events, rt, last = np.empty(n_te), [], [], 0.0, -10**9
    for k, s in enumerate(range(0, n_te, CHUNK)):
        a, b = i_va + s, min(i_va + s + CHUNK, n)
        p = pipe.predict(Xf.iloc[a:b]); preds[s:b - i_va] = p
        m = mean_absolute_error(yy.iloc[a:b], p); chunk_mae.append(m)
        if mode == "drift":      do = ph.update(m / T["ref_mae"]) and (k - last >= COOLDOWN)
        elif mode == "periodic": do = ((k + 1) % PERIODIC_EVERY == 0)
        else:                    do = False
        if do and b < n:                                                   # only labels fully observed (<= b-EMBARGO) are legal
            t0 = time.perf_counter(); pipe = make_models()[T["SEL"]]; l_ = max(0, b - EMBARGO - RECAL_WINDOW)
            pipe.fit(Xf.iloc[l_:b - EMBARGO], yy.iloc[l_:b - EMBARGO]); rt += time.perf_counter() - t0
            events.append(k); ph.reset(); last = k
    return dict(preds=preds, chunk_mae=np.array(chunk_mae), events=events, retrain_time=rt)

INIT_B = fit_initial(TB)
WF = {m: walk_forward(TB, m, INIT_B) for m in ["frozen", "periodic", "drift"]}
wf_table = pd.DataFrame([dict(mode=m, **metrics(TB["y_te"], r["preds"]), n_recalibrations=len(r["events"]),
                              retrain_seconds=r["retrain_time"]) for m, r in WF.items()])
display(wf_table.round(4)); wf_table.to_csv(f"{OUT_DIR}/table_closed_loop_modes.csv", index=False)
e_fro = np.abs(TB["y_te"] - WF["frozen"]["preds"])
for m in ["periodic", "drift"]:
    e_m = np.abs(TB["y_te"] - WF[m]["preds"]); lo, hi = block_bootstrap_ci(e_fro, e_m)
    print(f"{m} vs frozen: MAE change {100*(e_fro.mean()-e_m.mean())/e_fro.mean():+.2f}%  CI95 [{lo:+.2f}, {hi:+.2f}]  (positive = recalibration helps)")
fig, ax = plt.subplots(figsize=(11, 3.8))
for m, r in WF.items(): ax.plot(r["chunk_mae"], label=m, lw=1.2)
for k in WF["drift"]["events"]: ax.axvline(k, color="k", ls=":", alpha=.4)
ax.set_xlabel(f"cycle ({CHUNK} blocks)"); ax.set_ylabel("chunk MAE (tx/block)"); ax.set_title("Demand-forecast error, walk-forward, REAL confirmation period (2021-Q1) (dotted = recalibration)")
ax.legend(); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_closed_loop_mae.png", dpi=200); plt.show()
fig, ax = plt.subplots(figsize=(11, 3.8)); m_ = slice(0, min(1500, len(TB["y_te"])))
ax.plot(TB["y_te"][m_], label="actual demand", lw=.8); ax.plot(WF["drift"]["preds"][m_], label="forecast (closed loop)", lw=.8)
ax.set_xlabel("test block"); ax.set_ylabel("tx / block"); ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_demand_forecast.png", dpi=200); plt.show()
PREDS_CTRL = WF["drift"]["preds"]

In [ ]:
# 7) TRACE-DRIVEN QUEUE SIMULATOR
# World (identical for every policy):
#  * block times      = REAL inter-block intervals of the period
#  * arrival process  = Poisson with intensity proportional to the REAL demand (forward 6-block mean tx_count),
#                       scaled so the average offered load = rho x block capacity (operator-calibrated mean rate)
#  * tx attributes    = class (HIGH/NORMAL/LOW), size (lognormal), off-loadable flag; PRE-GENERATED per (rho, seed)
# Decision timing: the controller chooses its regime at the START of each interval, seeing only the backlog left after the
# previous block + (for ADAPTIVE) a forecast of the demand of the coming interval.  ORACLE sees the true arrival count.
REG = {
    "STATIC":   dict(order="fifo", elig=(),     B=1),    # executed baseline: FIFO, full payload on-chain
    "FAST":     dict(order="fifo", elig=(),     B=1),
    "FIFO_OFF": dict(order="fifo", elig=(1, 2), B=10),   # static hybrid storage, no priority
    "PRIO":     dict(order="prio", elig=(),     B=1),    # priority+aging only
    "MID":      dict(order="prio", elig=(2,),   B=10),
    "CONG":     dict(order="prio", elig=(1, 2), B=25),
}
FIXED_CANDIDATES = ["FIFO_OFF", "PRIO", "MID", "CONG"]

def make_world(sl, rel_hat_map):
    dt = df[TARGET_INT].iloc[sl].values.astype(float); dem = df[TARGET_DEM].iloc[sl].values.astype(float); sc = DEMAND_SCALE   # fixed TRAINING mean: no information from the evaluated period
    rh = {k: np.clip(np.asarray(v) / sc, 0.2, 3.0) for k, v in rel_hat_map.items()}
    rh["ADAPTIVE_CONST"] = np.ones(len(dt))          # ablation: knows only the AVERAGE load level, no time-varying forecast
    return dict(dt=dt, rel=dem / sc, rel_hat=rh)

def make_arrivals(world, rho, seed, S):
    rng = np.random.default_rng(seed); dt = world["dt"]
    n_arr = rng.poisson(rho * S["CAP"] / 600.0 * world["rel"] * dt); N = int(n_arr.sum())
    t_end = np.cumsum(dt); t_start = t_end - dt; blk = np.repeat(np.arange(len(dt)), n_arr)
    times = np.sort(t_start[blk] + rng.random(N) * dt[blk]); sg = S["SIZE_SIGMA"]
    return dict(t=times, cls=rng.choice(3, N, p=S["CLASS_P"]), size=rng.lognormal(-sg**2 / 2, sg, N),
                off=rng.random(N) < S["OFFLOADABLE_P"], offs=np.concatenate([[0], np.cumsum(n_arr)]), t_end=t_end)

def select_block(t, q_t, q_c, q_s, q_o, reg, S):
    nq = len(q_t)
    if nq == 0: return np.zeros(0, bool), 0.0
    if reg["order"] == "fifo": o = np.arange(nq)
    else: o = np.argsort(-(S["CLASS_W"][q_c] + (t - q_t) / S["T_AGE"]), kind="stable")
    elig = (np.isin(q_c, reg["elig"]) & q_o) if reg["elig"] else np.zeros(nq, bool)
    el_o, pos, so, to = elig[o], np.arange(nq), q_s[o], q_t[o]
    ne_pos, ne_cost = pos[~el_o], so[~el_o]; el_pos = pos[el_o]; m = len(el_pos); B = reg["B"]
    n_full = m // B; rem = m - n_full * B; nb = n_full
    if rem > 0 and (t - to[el_pos[n_full * B:]]).max() > S["MAX_WAIT_PARTIAL"]: nb += 1
    first = el_pos[::B][:nb] if nb > 0 else np.zeros(0, int)
    ranks = np.concatenate([ne_pos, first]); costs = np.concatenate([ne_cost, np.full(nb, S["ANCHOR"])])
    oi = np.argsort(ranks, kind="stable"); cum = np.cumsum(costs[oi]); k = int(np.searchsorted(cum, S["CAP"], side="right"))
    if k == 0: return np.zeros(nq, bool), 0.0
    inc = oi[:k]; is_ne = inc < len(ne_pos); inc_o = np.zeros(nq, bool); inc_o[ne_pos[inc[is_ne]]] = True
    if nb > 0:
        inc_b = np.zeros(nb, bool); inc_b[inc[~is_ne] - len(ne_pos)] = True
        bid = np.arange(m) // B; ok = bid < nb; inc_o[el_pos[ok][inc_b[bid[ok]]]] = True
    mask = np.zeros(nq, bool); mask[o[inc_o]] = True
    return mask, float(cum[k - 1])

def decide(policy, j, br0, arr, world, rho, S, P):
    """Regime chosen at the START of interval j (priority ordering is ALWAYS on; only the off-chain batching level adapts):
       s <= lo -> PRIO (no batching) | lo < s < hi -> MID | s >= hi -> CONG.
       FEEDBACK: s = backlog (in block capacities).  ADAPTIVE: backlog + forecast load.  ORACLE: backlog + TRUE future arrivals.
       Statistic mode 'I' = coming interval (s = backlog + load);  'H' = predicted backlog growth over the next H blocks."""
    if policy == "STATIC":     return "STATIC"
    if policy == "PRIO_ONLY":  return "PRIO"
    if policy == "BEST_FIXED": return P["BEST_FIXED"]
    if policy.startswith("FIXED:"): return policy.split(":")[1]
    H = S["H"]
    if policy == "FEEDBACK":
        s, (lo, hi) = br0, P["FEEDBACK"]
    else:
        key = policy if policy in ("ORACLE", "ADAPTIVE_CONST") else "ADAPTIVE"; lo, hi, mode = P[key]
        if policy == "ORACLE":
            nb_ = len(arr["offs"]) - 1
            if mode == "I": s = br0 + (arr["offs"][j + 1] - arr["offs"][j]) / S["CAP"]
            else: je = min(j + H, nb_); s = max(0.0, br0 + (arr["offs"][je] - arr["offs"][j]) / S["CAP"] - (je - j))
        else:
            u_hat = rho * world["rel_hat"][policy][j] * S["DT_MEAN"] / 600.0
            s = br0 + u_hat if mode == "I" else max(0.0, br0 + H * (u_hat - 1.0))
    return "CONG" if s >= hi else ("PRIO" if s <= lo else "MID")

def simulate(arr, world, policy, rho, S, P):
    nb = len(world["dt"]); offs = arr["offs"]
    q_t, q_c, q_s, q_o = np.empty(0), np.empty(0, int), np.empty(0), np.empty(0, bool)
    L, C, A, CT, used, backlog, counts = [], [], [], [], np.zeros(nb), np.zeros(nb), {}
    for j in range(nb):
        rname = decide(policy, j, len(q_t) / S["CAP"], arr, world, rho, S, P); counts[rname] = counts.get(rname, 0) + 1
        s, e = offs[j], offs[j + 1]
        if e > s:
            q_t = np.concatenate([q_t, arr["t"][s:e]]); q_c = np.concatenate([q_c, arr["cls"][s:e]])
            q_s = np.concatenate([q_s, arr["size"][s:e]]); q_o = np.concatenate([q_o, arr["off"][s:e]])
        t = arr["t_end"][j]; mask, u = select_block(t, q_t, q_c, q_s, q_o, REG[rname], S)
        if mask.any():
            L.append(t - q_t[mask]); C.append(q_c[mask]); A.append(q_t[mask]); CT.append(np.full(int(mask.sum()), t)); keep = ~mask
            q_t, q_c, q_s, q_o = q_t[keep], q_c[keep], q_s[keep], q_o[keep]
        used[j] = u; backlog[j] = len(q_t)
    cat = lambda l, d: np.concatenate(l) if l else np.empty(0, d)
    return dict(lat=cat(L, float), cls=cat(C, int), arr_t=cat(A, float), comp_t=cat(CT, float), arrivals_t=arr["t"], arrivals_cls=arr["cls"], used=used, backlog=backlog, q_t=q_t, q_c=q_c,
                t_end=arr["t_end"][-1], t_warm=arr["t_end"][int(S["WARMUP_FRAC"] * nb)], j_warm=int(S["WARMUP_FRAC"] * nb),
                n_arrived=len(arr["t"]), reg=counts)

def summarize(res, S, seg=0):
    """seg=0 -> whole period after warm-up; seg=1..4 -> consecutive quarters of it.
    Percentiles INCLUDE still-unfinished tx (their current wait = lower bound) so starvation cannot hide."""
    edges = np.linspace(res["t_warm"], res["t_end"], 5)
    lo, hi = (res["t_warm"], np.inf) if seg == 0 else (edges[seg - 1], edges[seg] if seg < 4 else np.inf)
    m = (res["arr_t"] >= lo) & (res["arr_t"] < hi); lat, cl = res["lat"][m], res["cls"][m]
    um = (res["q_t"] >= lo) & (res["q_t"] < hi); ua, uc = res["t_end"] - res["q_t"][um], res["q_c"][um]
    rows = []
    for name, sel in [("ALL", None), ("HIGH", 0), ("NORMAL", 1), ("LOW", 2)]:
        l = lat if sel is None else lat[cl == sel]; a = ua if sel is None else ua[uc == sel]; both = np.concatenate([l, a])
        pc = np.percentile(both, [50, 95, 99]) if len(both) else [np.nan] * 3
        rows.append(dict(cls=name, n_done=len(l), n_unfinished=len(a), mean=both.mean() if len(both) else np.nan,
                         std=both.std() if len(both) else np.nan, p50=pc[0], p95=pc[1], p99=pc[2],
                         max_wait=both.max() if len(both) else np.nan, sla_hit=(l <= S["SLA"]).sum() / max(len(both), 1)))
    bl = res["backlog"][res["j_warm"]:]; tot = sum(res["reg"].values())
    t_hi = res["t_end"] if not np.isfinite(hi) else hi; dur = max(t_hi - lo, 1.0)           # window [lo, t_hi)
    done_w = (res["comp_t"] >= lo) & (res["comp_t"] < t_hi) if np.isfinite(hi) else (res["comp_t"] >= lo)
    arr_w = (res["arrivals_t"] >= lo) & (res["arrivals_t"] < t_hi) if np.isfinite(hi) else (res["arrivals_t"] >= lo)
    comp_cls = res["cls"][done_w]; arr_cls = res["arrivals_cls"][arr_w]
    glob = dict(completed_tps=done_w.sum() / dur,                 # confirmations per second INSIDE the window (true completed throughput)
                arrival_tps=arr_w.sum() / dur,                    # offered arrival rate in the same window
                completed_to_arrived_ratio=done_w.sum() / max(arr_w.sum(), 1),
                completed_tps_HIGH=(comp_cls == 0).sum() / dur, completed_tps_NORMAL=(comp_cls == 1).sum() / dur, completed_tps_LOW=(comp_cls == 2).sum() / dur,
                arrival_tps_HIGH=(arr_cls == 0).sum() / dur, arrival_tps_NORMAL=(arr_cls == 1).sum() / dur, arrival_tps_LOW=(arr_cls == 2).sum() / dur,
                onchain_units_per_tx=res["used"].sum() / max(len(res["lat"]), 1), backlog_mean=bl.mean(),
                backlog_p95=np.percentile(bl, 95), backlog_max=bl.max(), backlog_final=res["backlog"][-1],
                frac_time_overloaded=float((bl > 5 * S["CAP"]).mean()),
                **{f"share_{k}": res["reg"].get(k, 0) / tot for k in ["STATIC", "FAST", "FIFO_OFF", "PRIO", "MID", "CONG"]})
    return rows, glob
print("Simulator ready.")

In [ ]:
# A1) DESIGN-FREEZE CHECK + load the controller settings that were tuned on VALIDATION in v5 (no re-tuning, no new simulation)
import ast
EXPECTED_HASH = json.load(open(f"{OUT_DIR}/RESULTS_SUMMARY.json"))["config_hash"]   # hash stored by the v5.2.1 main run
if not QUICK:
    assert CONFIG_HASH == EXPECTED_HASH, f"The design configuration changed! {CONFIG_HASH} != {EXPECTED_HASH}. Do NOT report results from this run."
print("Design hash:", CONFIG_HASH, "(identical to the v5.2.1 main run)" if CONFIG_HASH == EXPECTED_HASH else "(QUICK mode: differs by construction)")

def parse_params(p):
    try: return ast.literal_eval(str(p))
    except Exception: return str(p)
tune_df = pd.read_csv(f"{OUT_DIR}/table_tuning_validation.csv")
L_star = tune_df[tune_df.key.isin(["BEST_FIXED", "FEEDBACK", "ADAPTIVE", "ADAPTIVE_CONST"])].obj.min()
TUNED = {}
for key in ["BEST_FIXED", "FEEDBACK", "ADAPTIVE", "ADAPTIVE_CONST", "ORACLE"]:       # identical epsilon-constraint rule as v5
    pool = tune_df[tune_df.key == key]; ok = pool[pool.obj <= (1 + EPS_TOL) * L_star]
    best = ok.loc[ok.offload.idxmin()] if len(ok) else pool.loc[pool.obj.idxmin()]
    TUNED[key] = parse_params(best["params"])
print("Controller settings re-derived from the stored validation table:", TUNED)
summ_path = f"{OUT_DIR}/RESULTS_SUMMARY.json"
if os.path.exists(summ_path):
    st_ = json.load(open(summ_path))["tuned"]; norm = lambda v: tuple(v) if isinstance(v, (list, tuple)) else v
    same = all(norm(st_[k]) == norm(TUNED[k]) for k in TUNED)
    print("Matches the TUNED stored in RESULTS_SUMMARY.json:", same)

In [ ]:
# A2) REPRODUCIBILITY CHECK: rebuild the v5 test world and re-run a slice of the main experiment; compare with the stored results
rng_ = np.random.default_rng(7)
W_TEST = make_world(SL["te"], {"ADAPTIVE": WF["drift"]["preds"], "ADAPTIVE_FROZEN": WF["frozen"]["preds"],
                               "ADAPTIVE_SHUFFLED": rng_.permutation(WF["drift"]["preds"])})
repro_ok = None; rho_mid = RHOS[len(RHOS) // 2]
ref_path = f"{OUT_DIR}/sim_raw_results.csv"
if os.path.exists(ref_path):
    ref = pd.read_csv(ref_path); rho_chk = RHOS[0]; arr = make_arrivals(W_TEST, rho_chk, SEED + 0, SIM); rows_chk = []
    for pol in ["STATIC", "BEST_FIXED", "ADAPTIVE", "ORACLE"]:
        r_, g_ = summarize(simulate(arr, W_TEST, pol, rho_chk, SIM, TUNED), SIM); mine = next(r for r in r_ if r["cls"] == "ALL")
        old = ref[(ref.rho == rho_chk) & (ref.seed == 0) & (ref.policy == pol) & (ref.seg == 0) & (ref.cls == "ALL")].iloc[0]
        rows_chk.append(dict(policy=pol, p95_new=mine["p95"], p95_stored=old["p95"], mean_new=mine["mean"], mean_stored=old["mean"],
                             max_rel_diff=max(abs(mine["p95"] - old["p95"]) / old["p95"], abs(mine["mean"] - old["mean"]) / old["mean"])))
    repro = pd.DataFrame(rows_chk); display(repro.round(6)); repro.to_csv(f"{OUT_DIR}/table_reproducibility_check.csv", index=False)
    repro_ok = bool((repro.max_rel_diff < 1e-6).all())
    print(f"REPRODUCIBILITY (rho={rho_chk}, seed 0): {'PASS - identical to the stored v5 results' if repro_ok else 'DIFFERENCES FOUND (library versions / non-determinism) - inspect before reporting'}")
else:
    print("sim_raw_results.csv not found in OUT_DIR: reproducibility check skipped")

In [ ]:
# A3) ANALYTICAL VALIDATION OF THE SIMULATOR  (independent of the controller code)
# Theory.  At vanishing load every transaction is confirmed in the NEXT block, so its latency is the residual time to the next block.
# Arrivals in block-interval j are Poisson with intensity proportional to rel_j, uniform inside the interval.  Hence, exactly:
#       P(W > w) = sum_j rel_j * max(dt_j - w, 0) / sum_j rel_j * dt_j          (residual-life distribution of the REAL intervals)
# For exponential intervals with mean 600 s this gives Exp(600): p50 = 416 s, p95 = 1,797 s, p99 = 2,763 s.
def make_survival(dt, w):
    o = np.argsort(dt); d, ww = dt[o], w[o]
    suf_w = np.concatenate([np.cumsum(ww[::-1])[::-1], [0.0]]); suf_wd = np.concatenate([np.cumsum((ww * d)[::-1])[::-1], [0.0]]); tot = np.sum(w * dt)
    def surv(x):
        x = np.atleast_1d(np.asarray(x, float)); i = np.searchsorted(d, x, side="right")
        return (suf_wd[i] - x * suf_w[i]) / tot
    return surv, np.sum(w * dt**2 / 2) / tot
def theory_quantile(surv, q, hi):
    lo = 0.0
    for _ in range(80):
        mid = (lo + hi) / 2
        if 1 - surv(mid)[0] < q: lo = mid
        else: hi = mid
    return (lo + hi) / 2

RHO_LIGHT, qs = 0.02, [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
lat_all, jw = [], None
for s_ in range(3):
    arr = make_arrivals(W_TEST, RHO_LIGHT, SEED + 300 + s_, SIM); res = simulate(arr, W_TEST, "STATIC", RHO_LIGHT, SIM, TUNED)
    m = res["arr_t"] >= res["t_warm"]; lat_all.append(res["lat"][m]); jw = res["j_warm"]
lat = np.concatenate(lat_all)
surv, mean_th = make_survival(W_TEST["dt"][jw + 1:], W_TEST["rel"][jw + 1:])
rows = [dict(statistic="mean", theory_trace=mean_th, simulated=lat.mean(), exp600_ref=600.0)]
for q in qs: rows.append(dict(statistic=f"p{int(q*100)}", theory_trace=theory_quantile(surv, q, W_TEST["dt"].max()), simulated=np.quantile(lat, q), exp600_ref=-600 * np.log(1 - q)))
val = pd.DataFrame(rows); val["rel_error_pct"] = 100 * (val.simulated - val.theory_trace) / val.theory_trace
ks = sps.kstest(lat, lambda x: 1 - surv(x))
display(val.round(2)); val.to_csv(f"{OUT_DIR}/table_analytical_validation.csv", index=False)
# Pre-specified pass rule (sampling-noise aware): KS distance below the 1% critical value with a design effect of 2 (arrivals in the same
# interval share their confirmation time), AND mean within 2 %, AND median and p95 within 3 %.  The extreme quantiles p10/p99 are reported but
# not used for the decision because their sampling error is large.
ks_crit = 1.63 / np.sqrt(len(lat) / 2.0); e_ = val.set_index("statistic").rel_error_pct
analytic_ok = bool(ks.statistic <= ks_crit and abs(e_["mean"]) <= 2.0 and abs(e_["p50"]) <= 3.0 and abs(e_["p95"]) <= 3.0)
print(f"n = {len(lat):,} simulated latencies | KS distance D = {ks.statistic:.4f} (critical {ks_crit:.4f}) | mean err {e_['mean']:+.2f}% | p50 err {e_['p50']:+.2f}% | p95 err {e_['p95']:+.2f}%")
print("ANALYTICAL VALIDATION:", "PASS (simulator reproduces the exact residual-life distribution of the real block intervals)" if analytic_ok else "FAIL - do not trust the simulator")

# Saturation test: under heavy overload each block must be (almost) full, i.e. the simulator cannot serve more than the capacity and wastes < 3 %
sub = dict(dt=W_TEST["dt"][:1500], rel=W_TEST["rel"][:1500], rel_hat={k: v[:1500] for k, v in W_TEST["rel_hat"].items()})
arr = make_arrivals(sub, 1.5, SEED + 900, SIM); res = simulate(arr, sub, "STATIC", 1.5, SIM, TUNED)
fill = res["used"][200:].mean() / SIM["CAP"]; sat_ok = bool(fill >= 0.97)
print(f"Overload saturation (rho=1.5): mean block fill = {fill:.3f} of capacity -> {'PASS' if sat_ok else 'FAIL'};  final backlog = {res['backlog'][-1]:.0f} tx (grows without bound, as theory requires)")

In [ ]:
# A4) SENSITIVITY at rho = 0.5, 0.7 (and 0.85) over 11 assumption variants.  Controllers keep the v5 validation-tuned settings (no re-tuning).
SENS_LOADS = [0.5, 0.7, 0.85] if not QUICK else [0.5, 0.9]
n_s = 3 if not QUICK else 1
variants = [(f"offloadable={o}", dict(OFFLOADABLE_P=o)) for o in [0.2, 0.5, 0.8]]
variants += [("anchor=0.5", dict(ANCHOR=0.5)), ("anchor=2.0", dict(ANCHOR=2.0)),
             ("class mix 30/50/20", dict(CLASS_P=[0.30, 0.50, 0.20])), ("class mix 5/35/60", dict(CLASS_P=[0.05, 0.35, 0.60])),
             ("batch max-wait 600s", dict(MAX_WAIT_PARTIAL=600.0)), ("batch max-wait 2400s", dict(MAX_WAIT_PARTIAL=2400.0)),
             ("aging 900s", dict(T_AGE=900.0)), ("aging 3600s", dict(T_AGE=3600.0))]
SENS_POL = ["BEST_FIXED", "ADAPTIVE_CONST", "ADAPTIVE"]; rows = []; t0 = time.perf_counter()
for load in SENS_LOADS:
    for vname, mod in variants:
        S2 = dict(SIM, **mod); acc = {p: [] for p in SENS_POL}
        for sd in range(n_s):
            arr = make_arrivals(W_TEST, load, SEED + 700 + sd, S2)
            for pol in SENS_POL:
                rws, g_ = summarize(simulate(arr, W_TEST, pol, load, S2, TUNED), S2); p = {r["cls"]: r for r in rws}
                acc[pol].append(dict(macro=np.mean([p[c]["p95"] for c in CLS]), high=p["HIGH"]["p95"], low=p["LOW"]["p95"],
                                     sla=min(p[c]["sla_hit"] for c in CLS), off=1 - g_["onchain_units_per_tx"], ctps=g_["completed_tps"]))
        m = {pol: {k: np.mean([a[k] for a in v]) for k in v[0]} for pol, v in acc.items()}; BF, AD, CO = m["BEST_FIXED"], m["ADAPTIVE"], m["ADAPTIVE_CONST"]
        rows.append(dict(load=load, variant=vname, offchain_BEST_FIXED=BF["off"], offchain_ADAPTIVE=AD["off"],
                         offchain_saving_pct=100 * (BF["off"] - AD["off"]) / BF["off"],
                         macro_p95_cost_pct=100 * (AD["macro"] - BF["macro"]) / BF["macro"], HIGH_p95_cost_pct=100 * (AD["high"] - BF["high"]) / BF["high"],
                         LOW_p95_cost_pct=100 * (AD["low"] - BF["low"]) / BF["low"], SLA_gap_pp=100 * (AD["sla"] - BF["sla"]),
                         completed_tps_BEST_FIXED=BF["ctps"], completed_tps_ADAPTIVE=AD["ctps"], completed_tps_change_pct=100 * (AD["ctps"] - BF["ctps"]) / BF["ctps"],
                         ADAPT_vs_CONST_macro_gain_pct=100 * (CO["macro"] - AD["macro"]) / CO["macro"]))
    print(f"load {load} done ({time.perf_counter() - t0:.0f}s)")
sens = pd.DataFrame(rows); pd.set_option("display.width", 220); display(sens.round(2)); sens.to_csv(f"{OUT_DIR}/table_sensitivity_low_loads.csv", index=False)
rng_tab = sens.groupby("load")[["offchain_saving_pct", "macro_p95_cost_pct", "HIGH_p95_cost_pct", "LOW_p95_cost_pct", "SLA_gap_pp", "completed_tps_change_pct", "ADAPT_vs_CONST_macro_gain_pct"]].agg(["min", "max"]).round(2)
print("\nRANGE over all variants (cost > 0 means ADAPTIVE is WORSE than BEST_FIXED; saving > 0 means ADAPTIVE keeps more payload on-chain):"); display(rng_tab)
rng_tab.to_csv(f"{OUT_DIR}/table_sensitivity_ranges.csv")

In [ ]:
# A4b) COMPLETED THROUGHPUT UNDER OVERLOAD (rho >= 1).  Below rho = 1 every stable policy completes (almost) everything that arrives, so Completed TPS cannot
# separate policies; the throughput question is only meaningful when offered load reaches / exceeds the capacity.  Controllers keep the validation-tuned settings.
OV_LOADS = [1.0, 1.1, 1.25] if not QUICK else [1.1]
OV_POL = ["STATIC", "PRIO_ONLY", "BEST_FIXED", "FEEDBACK", "ADAPTIVE"]; n_o = 3 if not QUICK else 1; rows = []; t0 = time.perf_counter()
for load in OV_LOADS:
    for sd in range(n_o):
        arr = make_arrivals(W_TEST, load, SEED + 900 + sd, SIM)
        for pol in OV_POL:
            rws, g_ = summarize(simulate(arr, W_TEST, pol, load, SIM, TUNED), SIM); p = {r["cls"]: r for r in rws}
            rows.append(dict(load=load, seed=sd, policy=pol, completed_tps=g_["completed_tps"], arrival_tps=g_["arrival_tps"],
                             completed_to_arrived_ratio=g_["completed_to_arrived_ratio"], completed_tps_HIGH=g_["completed_tps_HIGH"],
                             completed_tps_NORMAL=g_["completed_tps_NORMAL"], completed_tps_LOW=g_["completed_tps_LOW"],
                             min_class_SLA=min(p[c]["sla_hit"] for c in CLS), HIGH_p95=p["HIGH"]["p95"], backlog_final=g_["backlog_final"],
                             offchain_share=1 - g_["onchain_units_per_tx"]))
    print(f"load {load} done ({time.perf_counter() - t0:.0f}s)")
ov = pd.DataFrame(rows).groupby(["load", "policy"]).mean(numeric_only=True).drop(columns="seed").reindex(OV_POL, level="policy")
pd.set_option("display.width", 220); display(ov.round(4)); ov.to_csv(f"{OUT_DIR}/table_completed_tps_overload.csv")
print("Completed TPS = confirmations per second inside the evaluation window.  At rho >= 1 a policy that cannot keep up shows completed_to_arrived_ratio < 1 and a growing backlog_final.")

In [ ]:
# A4c) MATCHED OFF-CHAIN FRONTIER (answers: does the forecast help at the SAME off-chain reliance?)  POST-HOC, DESCRIPTIVE: this sweep runs on the confirmation period,
# it is NOT used to select any setting.  Policies at different thresholds trade latency against off-chain share, so a fair comparison fixes the off-chain share.
FR_LOADS = [0.5, 0.7] if not QUICK else [0.5]; n_f = 2 if not QUICK else 1
combos = [(lo, hi) for lo in GRID_LO for hi in GRID_HI if hi > lo]
FR_POL = ["ADAPTIVE", "ADAPTIVE_CONST", "ADAPTIVE_SHUFFLED", "FEEDBACK"]; rows = []; t0 = time.perf_counter()
for load in FR_LOADS:
    for sd in range(n_f):
        arr = make_arrivals(W_TEST, load, SEED + 1300 + sd, SIM)
        for pol in FR_POL:
            for lo, hi in combos:
                P2 = dict(TUNED)
                if pol == "FEEDBACK": P2["FEEDBACK"] = (lo, hi)
                elif pol == "ADAPTIVE_CONST": P2["ADAPTIVE_CONST"] = (lo, hi, "I")
                else: P2["ADAPTIVE"] = (lo, hi, "I")
                rws, g_ = summarize(simulate(arr, W_TEST, pol, load, SIM, P2), SIM)
                rows.append(dict(load=load, seed=sd, policy=pol, lo=lo, hi=hi, macro_p95=np.mean([r["p95"] for r in rws if r["cls"] in CLS]),
                                 offchain_share=1 - g_["onchain_units_per_tx"], completed_tps=g_["completed_tps"]))
    print(f"load {load} done ({time.perf_counter() - t0:.0f}s)")
fr = pd.DataFrame(rows).groupby(["load", "policy", "lo", "hi"], as_index=False).mean(numeric_only=True).drop(columns="seed")
fr.to_csv(f"{OUT_DIR}/table_frontier_POSTHOC.csv", index=False)
def envelope(d):                       # lowest macro p95 achievable at or below a given off-chain share (running minimum along increasing off-chain)
    d = d.sort_values("offchain_share"); return d.offchain_share.values, np.minimum.accumulate(d.macro_p95.values)
out = []
for load in FR_LOADS:
    env = {p: envelope(fr[(fr.load == load) & (fr.policy == p)]) for p in FR_POL}
    lo_ = max(e[0].min() for e in env.values()); hi_ = min(e[0].max() for e in env.values())
    for level in np.linspace(lo_, hi_, 5):
        v = {p: float(np.interp(level, e[0], e[1])) for p, e in env.items()}
        out.append(dict(load=load, offchain_share=level, **{f"p95_{p}": v[p] for p in FR_POL},
                        ADAPT_vs_CONST_pct=100 * (v["ADAPTIVE_CONST"] - v["ADAPTIVE"]) / v["ADAPTIVE_CONST"],
                        ADAPT_vs_SHUFFLED_pct=100 * (v["ADAPTIVE_SHUFFLED"] - v["ADAPTIVE"]) / v["ADAPTIVE_SHUFFLED"],
                        ADAPT_vs_FEEDBACK_pct=100 * (v["FEEDBACK"] - v["ADAPTIVE"]) / v["FEEDBACK"]))
mf = pd.DataFrame(out); display(mf.round(3)); mf.to_csv(f"{OUT_DIR}/table_matched_offchain_POSTHOC.csv", index=False)
print("Positive = ADAPTIVE has LOWER class-averaged p95 than the comparator AT THE SAME off-chain share.  Gains near 0 (inside seed noise) = the forecast adds no demonstrable value beyond backlog feedback.")

In [ ]:
# A4d) HELPERS for the extra stress experiments.  The MAIN simulator, controller and tuned settings are NOT modified (design hash unchanged);
# simulate_ex is a copy of simulate() with two optional extras: (i) a per-block capacity multiplier (unannounced capacity dip) and (ii) "shadow" decisions
# (what another policy WOULD have chosen in the same state), used for the decision-divergence diagnostic.  With no extras it equals simulate().
from scipy import stats as _st
RIDX = {k: i for i, k in enumerate(REG)}
def simulate_ex(arr, world, policy, rho, S, P, cap_mult=None, shadow=()):
    nb = len(world["dt"]); offs = arr["offs"]
    q_t, q_c, q_s, q_o = np.empty(0), np.empty(0, int), np.empty(0), np.empty(0, bool)
    L, C, A, CT, used, backlog, counts = [], [], [], [], np.zeros(nb), np.zeros(nb), {}
    sh = {p: np.zeros(nb, int) for p in shadow}; regs = np.zeros(nb, int)
    for j in range(nb):
        br0 = len(q_t) / S["CAP"]; rname = decide(policy, j, br0, arr, world, rho, S, P); counts[rname] = counts.get(rname, 0) + 1; regs[j] = RIDX[rname]
        for p in shadow: sh[p][j] = RIDX[decide(p, j, br0, arr, world, rho, S, P)]
        s, e = offs[j], offs[j + 1]
        if e > s:
            q_t = np.concatenate([q_t, arr["t"][s:e]]); q_c = np.concatenate([q_c, arr["cls"][s:e]])
            q_s = np.concatenate([q_s, arr["size"][s:e]]); q_o = np.concatenate([q_o, arr["off"][s:e]])
        Sj = S if cap_mult is None else dict(S, CAP=S["CAP"] * float(cap_mult[j]))
        t = arr["t_end"][j]; mask, u = select_block(t, q_t, q_c, q_s, q_o, REG[rname], Sj)
        if mask.any():
            L.append(t - q_t[mask]); C.append(q_c[mask]); A.append(q_t[mask]); CT.append(np.full(int(mask.sum()), t)); keep = ~mask
            q_t, q_c, q_s, q_o = q_t[keep], q_c[keep], q_s[keep], q_o[keep]
        used[j] = u; backlog[j] = len(q_t)
    cat = lambda l, d: np.concatenate(l) if l else np.empty(0, d)
    return dict(lat=cat(L, float), cls=cat(C, int), arr_t=cat(A, float), comp_t=cat(CT, float), arrivals_t=arr["t"], arrivals_cls=arr["cls"], used=used, backlog=backlog,
                q_t=q_t, q_c=q_c, t_end=arr["t_end"][-1], t_warm=arr["t_end"][int(S["WARMUP_FRAC"] * nb)], j_warm=int(S["WARMUP_FRAC"] * nb),
                n_arrived=len(arr["t"]), reg=counts, regs=regs, shadow=sh)

# equivalence check: simulate_ex(no extras) must reproduce simulate() exactly
_arr = make_arrivals(W_TEST, 0.7, SEED + 11, SIM); _a = simulate(_arr, W_TEST, "ADAPTIVE", 0.7, SIM, TUNED); _b = simulate_ex(_arr, W_TEST, "ADAPTIVE", 0.7, SIM, TUNED)
ex_ok = bool(np.array_equal(_a["lat"], _b["lat"]) and np.array_equal(_a["backlog"], _b["backlog"]))
print("simulate_ex == simulate (no extras):", "PASS" if ex_ok else "FAIL - do not use the stress tables"); assert ex_ok

NB_T = len(W_TEST["dt"]); ARR_TEND = np.cumsum(W_TEST["dt"])
def periodic(nb, period, offset, length):
    m = np.zeros(nb, bool); ends = []
    for s_ in range(offset, nb, period):
        m[s_:s_ + length] = True; ends.append(min(s_ + length, nb - 1))
    return m, ends
def blocks_of(times): return np.searchsorted(ARR_TEND, times, side="right").clip(0, NB_T - 1)
def window_stats(res, mask_blocks):
    # class-wise p95 of tx ARRIVING inside the window (completed + still-waiting; unfinished waits are a lower bound)
    done = mask_blocks[blocks_of(res["arr_t"])]; un = mask_blocks[blocks_of(res["q_t"])]; out = {}
    for c_, nm in enumerate(CLS):
        w = np.concatenate([res["lat"][done & (res["cls"] == c_)], res["t_end"] - res["q_t"][un & (res["q_c"] == c_)]])
        out[nm] = np.percentile(w, 95) if len(w) else np.nan
    out["macro"] = np.nanmean([out[n] for n in CLS]); return out
def recovery_blocks(res, ends, limit=500):
    # blocks after the event ends until the backlog is <= 1 block of capacity (censored at `limit`)
    bl = res["backlog"]; rec = []
    for e in ends:
        if e < res["j_warm"]: continue
        idx = np.where(bl[e:e + limit] <= SIM["CAP"])[0]; rec.append(idx[0] if len(idx) else limit)
    return (float(np.mean(rec)) if rec else np.nan), (float(np.mean(np.array(rec) >= limit)) if rec else np.nan)
def paired(df, key, metric, a, b, pct=True):
    # seed-paired difference (b - a), relative to b if pct; positive = policy a is lower/better.  95% t-interval over seeds.
    out = []
    for k, g in df.groupby(key):
        pa = g[g.policy == a].set_index("seed")[metric]; pb = g[g.policy == b].set_index("seed")[metric]; d = (pb - pa).dropna()
        d = 100 * d / pb.loc[d.index] if pct else d; h = _st.t.ppf(0.975, max(len(d) - 1, 1)) * d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else np.nan
        out.append(dict(**(dict(zip(key, k if isinstance(k, tuple) else (k,)))), comparator=b, metric=metric, a_better_by=d.mean(), ci95=h, n=len(d)))
    return pd.DataFrame(out)
STRESS_POL = ["STATIC", "BEST_FIXED", "FEEDBACK", "ADAPTIVE_CONST", "ADAPTIVE"]
def one_run(arr, world, pol, rho, **kw):
    res = simulate_ex(arr, world, pol, rho, SIM, TUNED, **kw); rws, g_ = summarize(res, SIM); p = {r["cls"]: r for r in rws}
    return res, dict(macro_p95=np.mean([p[c]["p95"] for c in CLS]), HIGH_p95=p["HIGH"]["p95"], LOW_p95=p["LOW"]["p95"], p99_all=p["ALL"]["p99"],
                     min_class_SLA=min(p[c]["sla_hit"] for c in CLS), backlog_max=g_["backlog_max"], offchain_share=1 - g_["onchain_units_per_tx"], completed_tps=g_["completed_tps"])

In [ ]:
# A4e) BURSTY ERP WORKLOAD (synthetic stress scenarios on top of the REAL block trace; base offered load rho = 0.6; no re-tuning)
#   S1 order-processing bursts: x2.5 for 36 blocks (~6 h) every 1008 blocks (~weekly), NOT announced to the forecaster
#   S2 month-end close: x1.7 for 288 blocks (~2 days) at the end of every 4320-block "month", announced to the forecaster (the schedule is known in advance)
#   S3 same month-end close but NOT announced (isolates what knowing the schedule is worth)
# Burst schedules are ASSUMED workload shapes (not measured ERP traces) - say so in the paper.
RHO_B = 0.6; n_b = 5 if not QUICK else 1
m1, e1 = periodic(NB_T, 1008, 500, 36); m2, e2 = periodic(NB_T, 4320, 4000, 288)
SCEN = {"S1 sudden bursts (unannounced)": (1 + 1.5 * m1, False, m1, e1), "S2 month-end close (forecast informed)": (1 + 0.7 * m2, True, m2, e2),
        "S3 month-end close (unannounced)": (1 + 0.7 * m2, False, m2, e2)}
rows = []; t0 = time.perf_counter()
for sname, (prof, known, mk, ends) in SCEN.items():
    Wb = dict(dt=W_TEST["dt"], rel=W_TEST["rel"] * prof, rel_hat={k: (np.clip(v * prof, 0.2, 6.0) if (known and k != "ADAPTIVE_CONST") else v) for k, v in W_TEST["rel_hat"].items()})
    aft = mk.copy()
    for e_ in ends: aft[e_:e_ + 144] = True                         # burst + 144 blocks (~1 day) of aftermath
    for sd in range(n_b):
        arr = make_arrivals(Wb, RHO_B, SEED + 2100 + sd, SIM)
        for pol in STRESS_POL:
            res, m = one_run(arr, Wb, pol, RHO_B); ws = window_stats(res, aft); rc, cens = recovery_blocks(res, ends)
            rows.append(dict(scenario=sname, seed=sd, policy=pol, **m, window_macro_p95=ws["macro"], window_HIGH_p95=ws["HIGH"], window_LOW_p95=ws["LOW"], recovery_blocks=rc, not_recovered_share=cens))
    print(f"{sname} done ({time.perf_counter() - t0:.0f}s)")
bt = pd.DataFrame(rows); bt.to_csv(f"{OUT_DIR}/table_bursty_raw.csv", index=False)
summ = bt.groupby(["scenario", "policy"]).mean(numeric_only=True).drop(columns="seed").reindex(STRESS_POL, level="policy"); pd.set_option("display.width", 250); display(summ.round(2)); summ.to_csv(f"{OUT_DIR}/table_bursty.csv")
pc = pd.concat([paired(bt, ["scenario"], m_, "ADAPTIVE", b_, pct=(m_ != "recovery_blocks")) for b_ in ["BEST_FIXED", "FEEDBACK", "ADAPTIVE_CONST"]
                for m_ in ["window_macro_p95", "window_HIGH_p95", "macro_p95", "offchain_share", "recovery_blocks"]])
display(pc.round(2)); pc.to_csv(f"{OUT_DIR}/table_bursty_paired.csv", index=False)
print("a_better_by > 0: ADAPTIVE is lower than the comparator (% for latency/off-chain, blocks for recovery).  CI = seed-paired 95% t-interval.")

In [ ]:
# A4f) UNANNOUNCED CAPACITY DIP WITH RECOVERY.  A constant capacity cut equals a higher offered load (rho/c) and is already covered by A4b, so the NEW question is a
# TEMPORARY drop (72 blocks ~12 h every 1008 blocks) followed by return to normal: tail latency during the dip + aftermath and the time needed to clear the backlog.
# The controller is NOT told about the dip (it only sees its backlog).  No re-tuning.
DIP_C = [0.8, 0.6, 0.4] if not QUICK else [0.4]; DIP_RHO = [0.5, 0.7] if not QUICK else [0.5]; n_d = 4 if not QUICK else 1
md_, ed_ = periodic(NB_T, 1008, 500, 72); aft = md_.copy()
for e_ in ed_: aft[e_:e_ + 144] = True
rows = []; t0 = time.perf_counter()
for c_ in DIP_C:
    cm = np.where(md_, c_, 1.0)
    for rho in DIP_RHO:
        for sd in range(n_d):
            arr = make_arrivals(W_TEST, rho, SEED + 2600 + sd, SIM)
            for pol in STRESS_POL:
                res, m = one_run(arr, W_TEST, pol, rho, cap_mult=cm); ws = window_stats(res, aft); rc, cens = recovery_blocks(res, ed_)
                rows.append(dict(capacity_during_dip=c_, rho=rho, seed=sd, policy=pol, **m, window_macro_p95=ws["macro"], window_HIGH_p95=ws["HIGH"], recovery_blocks=rc, not_recovered_share=cens))
    print(f"capacity {c_} done ({time.perf_counter() - t0:.0f}s)")
dp = pd.DataFrame(rows); dp.to_csv(f"{OUT_DIR}/table_capacity_dip_raw.csv", index=False)
sm = dp.groupby(["capacity_during_dip", "rho", "policy"]).mean(numeric_only=True).drop(columns="seed").reindex(STRESS_POL, level="policy"); display(sm.round(2)); sm.to_csv(f"{OUT_DIR}/table_capacity_dip.csv")
pc = pd.concat([paired(dp, ["capacity_during_dip", "rho"], m_, "ADAPTIVE", b_, pct=(m_ != "recovery_blocks")) for b_ in ["BEST_FIXED", "FEEDBACK"]
                for m_ in ["window_macro_p95", "window_HIGH_p95", "offchain_share", "recovery_blocks"]])
display(pc.round(2)); pc.to_csv(f"{OUT_DIR}/table_capacity_dip_paired.csv", index=False)
print("Recovery = blocks until the backlog is <= 1 block of capacity after the dip ends (500 = not recovered before the next event).")

In [ ]:
# A4g) PRIORITY FAIRNESS / STARVATION (confirmation-period simulation; same arrivals for every policy).
# Priority scheduling is unequal BY DESIGN, so a Jain index on raw latency would mislead.  We report (a) per-class p95, SLA and maximum wait, (b) STARVATION RATE = share of a
# class's tx (completed or still waiting) that waited longer than 3 x SLA, (c) Jain's index of the three classes' SLA attainment (1 = equal service, 1/3 = one class gets all),
# (d) the HIGH-over-LOW p95 ratio.
FAIR_POL = ["STATIC", "PRIO_ONLY", "BEST_FIXED", "FEEDBACK", "ADAPTIVE"]; n_f2 = 3 if not QUICK else 1; STARVE = 3 * SIM["SLA"]; rows = []; t0 = time.perf_counter()
jain = lambda x: float(np.sum(x) ** 2 / (len(x) * np.sum(np.square(x)))) if np.sum(np.square(x)) > 0 else np.nan
for rho in RHOS:
    for sd in range(n_f2):
        arr = make_arrivals(W_TEST, rho, SEED + 3100 + sd, SIM)
        for pol in FAIR_POL:
            res = simulate_ex(arr, W_TEST, pol, rho, SIM, TUNED); j0 = res["t_warm"]; mk = res["arr_t"] >= j0; mu = res["q_t"] >= j0; rec = dict(rho=rho, seed=sd, policy=pol); sl = []
            for c_, nm in enumerate(CLS):
                done = res["lat"][mk & (res["cls"] == c_)]; w = np.concatenate([done, res["t_end"] - res["q_t"][mu & (res["q_c"] == c_)]])
                rec[f"{nm}_p95"] = np.percentile(w, 95); rec[f"{nm}_max_wait"] = w.max(); rec[f"{nm}_starvation"] = float((w > STARVE).mean()); rec[f"{nm}_SLA"] = float((done <= SIM["SLA"]).sum() / len(w)); sl.append(rec[f"{nm}_SLA"])
            rec["jain_SLA"] = jain(sl); rec["min_class_SLA"] = min(sl); rec["HIGH_over_LOW_p95"] = rec["HIGH_p95"] / rec["LOW_p95"]; rows.append(rec)
    print(f"rho {rho} done ({time.perf_counter() - t0:.0f}s)")
fa = pd.DataFrame(rows); fs = fa.groupby(["rho", "policy"]).mean(numeric_only=True).drop(columns="seed").reindex(FAIR_POL, level="policy")
display(fs.round(3)); fs.to_csv(f"{OUT_DIR}/table_fairness.csv"); fa.to_csv(f"{OUT_DIR}/table_fairness_raw.csv", index=False)
pf = pd.concat([paired(fa, ["rho"], m_, "ADAPTIVE", "BEST_FIXED", pct=False) for m_ in ["LOW_p95", "LOW_starvation", "jain_SLA", "min_class_SLA"]]); display(pf.round(4)); pf.to_csv(f"{OUT_DIR}/table_fairness_paired.csv", index=False)
print("Reading guide: LOW starvation / LOW max wait show whether protecting HIGH starves LOW.  a_better_by > 0 on LOW_p95 / LOW_starvation = ADAPTIVE is LOWER (better) than BEST_FIXED; on jain/min-class SLA it means LOWER.")

In [ ]:
# A4h) DOES THE FORECAST ACTUALLY CHANGE THE DECISIONS?  Run ADAPTIVE and, at every block and in the SAME state (same backlog), ask what the no-forecast ablation (CONST), the
# shuffled-forecast placebo and the backlog-only FEEDBACK controller would have chosen.  divergence = share of blocks where the choice differs from ADAPTIVE's.
DV_POL = ["ADAPTIVE_CONST", "ADAPTIVE_SHUFFLED", "FEEDBACK"]; n_v = 3 if not QUICK else 1; rows = []
for rho in RHOS:
    for sd in range(n_v):
        arr = make_arrivals(W_TEST, rho, SEED + 3600 + sd, SIM); res = simulate_ex(arr, W_TEST, "ADAPTIVE", rho, SIM, TUNED, shadow=DV_POL); j0 = res["j_warm"]
        rec = dict(rho=rho, seed=sd)
        for p in DV_POL: rec[f"diverge_vs_{p}"] = float((res["shadow"][p][j0:] != res["regs"][j0:]).mean())
        for rn in ["PRIO", "MID", "CONG"]: rec[f"ADAPTIVE_share_{rn}"] = float((res["regs"][j0:] == RIDX[rn]).mean())
        rows.append(rec)
dv = pd.DataFrame(rows).groupby("rho").mean().drop(columns="seed"); display(dv.round(3)); dv.to_csv(f"{OUT_DIR}/table_decision_divergence.csv")
print("Small divergence vs CONST / SHUFFLED = the forecast term rarely changes the regime, which explains why its measured value is small.")

In [ ]:
# A5) FIG 5 and FIG 6 FOR THE MODELS THAT WERE ACTUALLY SELECTED (selection was on validation; diagnostics here use held-out data, no refitting)
from sklearn.inspection import permutation_importance
def perm_imp(T):
    Xv, yv = df[T["feats"]].iloc[SL["va"]], df[T["target"]].iloc[SL["va"]]
    r = permutation_importance(T["fitted"][T["SEL"]], Xv, yv, n_repeats=5, random_state=SEED, scoring="neg_mean_absolute_error", n_jobs=1)
    return pd.DataFrame(dict(feature=T["feats"], importance_MAE_increase=r.importances_mean, std=r.importances_std)).sort_values("importance_MAE_increase")
pi_B, pi_A = perm_imp(TB), perm_imp(TA)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, pi, T, ttl in [(axes[0], pi_B, TB, f"Task B - demand forecast ({TB['SEL']})"), (axes[1], pi_A, TA, f"Task A - block interval ({TA['SEL']}) : negative control")]:
    ax.barh(pi.feature, pi.importance_MAE_increase, xerr=pi["std"], color=["#c0392b" if v >= pi.importance_MAE_increase.nlargest(2).min() else "#2c6fbb" for v in pi.importance_MAE_increase])
    ax.set_title(ttl); ax.set_xlabel("increase in validation MAE when the feature is permuted")
plt.suptitle("Fig 5 (revised): permutation importance of the SELECTED models (validation period)"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig5_permutation_importance_selected_models.png", dpi=200); plt.show()
pd.concat([pi_B.assign(task="B demand"), pi_A.assign(task="A interval")]).to_csv(f"{OUT_DIR}/table_permutation_importance.csv", index=False)
print("Top-3 features, demand model:", list(pi_B.feature[::-1][:3])); print("Top-3 features, interval model:", list(pi_A.feature[::-1][:3]), "(importances ~ 0 => nothing to explain: the interval is unpredictable)")

def acf(x, nl=48): x = np.asarray(x, float) - np.mean(x); d_ = np.sum(x * x); return np.array([np.sum(x[k:] * x[:-k]) / d_ for k in range(1, nl + 1)])
fig, axes = plt.subplots(2, 2, figsize=(13, 9)); pB, rB = TB["PT"][TB["SEL"]], TB["y_te"] - TB["PT"][TB["SEL"]]; pA, rA = TA["PT"][TA["SEL"]], TA["y_te"] - TA["PT"][TA["SEL"]]
axes[0, 0].scatter(pB, rB, s=3, alpha=.25); axes[0, 0].axhline(0, color="k", lw=.8); axes[0, 0].set_title(f"(a) demand: residual vs predicted ({TB['SEL']})"); axes[0, 0].set_xlabel("predicted tx/block"); axes[0, 0].set_ylabel("residual")
axes[0, 1].hist(rB, bins=60); axes[0, 1].set_title("(b) demand: residual distribution"); axes[0, 1].set_xlabel("residual")
a_ = acf(rB); axes[1, 0].bar(range(1, 49), a_); axes[1, 0].axhline(1.96 / np.sqrt(len(rB)), color="r", ls="--"); axes[1, 0].axhline(-1.96 / np.sqrt(len(rB)), color="r", ls="--")
axes[1, 0].set_title("(c) demand: residual autocorrelation (red = 95% white-noise band)"); axes[1, 0].set_xlabel("lag (blocks)")
axes[1, 1].scatter(pA, rA, s=3, alpha=.25); axes[1, 1].axhline(0, color="k", lw=.8); axes[1, 1].set_title(f"(d) interval: residual vs predicted ({TA['SEL']}) - negative control"); axes[1, 1].set_xlabel("predicted seconds")
plt.suptitle("Fig 6 (revised): residual diagnostics of the SELECTED models (2021-Q1 test period)"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig6_residual_diagnostics_selected_models.png", dpi=200); plt.show()
print(f"Demand model: bias = {rB.mean():+.1f} tx/block, residual std = {rB.std():.1f}, corr(|res|, pred) = {np.corrcoef(np.abs(rB), pB)[0,1]:+.2f}, lag-1 ACF = {a_[0]:+.2f}")
print("Ljung-Box (demand residuals):", json.dumps(ljung_box(rB)))
print(f"Interval model: prediction std = {pA.std():.1f}s vs target std = {TA['y_te'].std():.1f}s (a near-constant prediction = no information)")

In [ ]:
# A6) POST-HOC / EXPLORATORY: value of information (ORACLE headroom).  NOT a pre-specified claim: label it "exploratory" in the paper.
if os.path.exists(ref_path):
    d = pd.read_csv(ref_path); d = d[d.seg == 0]; rows = []
    for rho in sorted(d.rho.unique()):
        for pol in ["BEST_FIXED", "ADAPTIVE", "ORACLE"]:
            y = d[(d.rho == rho) & (d.policy == pol)]; cl = y[y.cls.isin(CLS)]; g = y[y.cls == "ALL"]
            rows.append(dict(rho=rho, policy=pol, offchain_share=1 - g.onchain_units_per_tx.mean(), macro_p95=cl.groupby("seed").p95.mean().mean(),
                             HIGH_p95=y[y.cls == "HIGH"].p95.mean(), min_class_SLA=cl.groupby("seed").sla_hit.min().mean()))
    voi = pd.DataFrame(rows)
    out = []
    for rho in sorted(voi.rho.unique()):
        v = voi[voi.rho == rho].set_index("policy")
        for pol in ["ADAPTIVE", "ORACLE"]:
            out.append(dict(rho=rho, policy=pol, offchain_saved_vs_BEST_FIXED_pct=100 * (v.loc["BEST_FIXED", "offchain_share"] - v.loc[pol, "offchain_share"]) / v.loc["BEST_FIXED", "offchain_share"],
                            macro_p95_cost_vs_BEST_FIXED_pct=100 * (v.loc[pol, "macro_p95"] - v.loc["BEST_FIXED", "macro_p95"]) / v.loc["BEST_FIXED", "macro_p95"],
                            SLA_gap_pp=100 * (v.loc[pol, "min_class_SLA"] - v.loc["BEST_FIXED", "min_class_SLA"])))
    voi_tab = pd.DataFrame(out); display(voi.round(3)); display(voi_tab.round(2)); voi_tab.to_csv(f"{OUT_DIR}/table_value_of_information_POSTHOC.csv", index=False)
    print("EXPLORATORY. ORACLE (perfect knowledge of the next interval's arrivals) could keep far more payload on-chain at a small latency cost, "
          "while the demand forecast captures almost none of that headroom: the remaining uncertainty is the (unpredictable) block interval.")
    print("NOTE: the scorecard row 'Gap to ORACLE is small' is satisfied for an uninformative reason (ORACLE is tuned to use LESS off-chain, so it is not a latency upper bound). Use this table instead.")
else:
    print("sim_raw_results.csv not found: skipped")

In [ ]:
# A7) SAVE + SUMMARY OF CHECKS
checks = dict(design_hash=CONFIG_HASH, reproducibility_vs_v5=repro_ok, analytical_validation=analytic_ok, saturation_test=sat_ok)
print(json.dumps(checks, indent=2))
summary = dict(checks=checks, analytic_table=val.round(3).to_dict("records"), ks_D=float(ks.statistic),
               sensitivity_ranges=json.loads(rng_tab.to_json()), top_features_demand=list(pi_B.feature[::-1][:5]))
json.dump(summary, open(f"{OUT_DIR}/ADDENDUM_SUMMARY.json", "w"), indent=2, default=str)
import shutil; shutil.make_archive("/content/results_v5_2_1_addendum", "zip", OUT_DIR)
print("Saved:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files; files.download("/content/results_v5_2_1_addendum.zip")
except Exception: pass